<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Deep network or gradient boosting?

**[Deep network or gradient boosting?](https://learning.nextia-ai.com/courses/deep-learning/m09/deep-network-or-gradient-boosting/)** · Deep Learning and Transformers Explained · Module 9, case study 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare a neural network with gradient boosting on a real table of credit-card clients, under one split and one metric: the score, the spread across seeds, the preparation, the tuning effort, the training time and the explanations. Then you decide whether the network is worth it.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 2 minutes on a CPU (about 70 small training runs). |
| **Needs** | An internet connection for the setup cells. No account. PyTorch, pandas and scikit-learn (preinstalled in Colab and Kaggle). |
| **You should know** | The training loop, early stopping and dropout from Modules 3 and 4 of this course ([The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/), [Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/)). The credit case study of the machine learning course ([Predict credit-card default](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/)) helps, but is not required. |
| **Data** | UCI *Default of Credit Card Clients* (Yeh, 2009), CC BY 4.0, https://doi.org/10.24432/C55S3H. Real, anonymous clients of one bank in Taiwan, 2005. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m09/deep-network-or-gradient-boosting/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M09-L02-deep-network-or-gradient-boosting/deep-network-or-gradient-boosting-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: get the data

Run the next cell. It downloads `credit_default_taiwan.csv.gz` (about 1 MB) from the Nextia labs repository into a folder `credit-network` and checks its checksum. It needs an internet connection and no account. If a copy of the file is next to this notebook, the cell uses that copy.

You should see `Ready: credit_default_taiwan.csv.gz, checksum OK.`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `credit-network` (in Colab: the **Files** panel on the left), then run the cell again.

The data is real and anonymous: 30,000 credit-card clients of one bank in Taiwan in 2005. Data: "Default of Credit Card Clients" by I-Cheng Yeh, UCI Machine Learning Repository, https://doi.org/10.24432/C55S3H, licensed under CC BY 4.0. Nextia Learning converted it from XLS to a gzip CSV, removed the extra first header row and renamed the target column to `default`; nothing else changed.

In [ ]:
# Setup: download the data file into credit-network/ and check it. Standard library only.
import hashlib, os, shutil, subprocess, urllib.parse, urllib.request
from pathlib import Path

BASE = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
URL = urllib.parse.urljoin(BASE, "../C05/case-studies/data/credit_default_taiwan.csv.gz")
FILE, SHA256 = "credit_default_taiwan.csv.gz", "a773cfc32f6759261cebfdfdc4f744bce1b247a583fe7c06363a12702c98f727"
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("credit-network").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

target = WORK / FILE
if not (target.exists() and sha256(target) == SHA256):
    if (HERE / FILE).exists():
        shutil.copy(HERE / FILE, target)
    else:
        try:
            urllib.request.urlretrieve(URL, target)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(target), URL]).returncode:
                raise SystemExit(f"Could not download {URL}. Check your internet connection.")
if sha256(target) != SHA256:
    raise SystemExit(f"{FILE} has the wrong checksum. Delete the folder credit-network and run again.")
os.chdir(WORK)
print(f"Ready: {FILE}, checksum OK.")

## Setup: threads

The next cell gives PyTorch and scikit-learn the same two CPU threads, so that the training times compare like with like. Run it. You should see `PyTorch threads: 2`.

In [ ]:
# Both models get 2 CPU threads, so their times compare fairly.
import torch
from torch import nn
from threadpoolctl import threadpool_limits
torch.set_num_threads(2)
threadpool_limits(2)
print("PyTorch threads:", torch.get_num_threads())

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the split

The bank's card-risk team wants to know which clients will miss next month's payment, so that it can offer them an early-help call. The machine learning course's [credit case study](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/) chose logistic regression. A colleague now asks: would a neural network do better?

A fair contest starts with the same data, the same split and the same metric as before. The next cell repeats the machine learning course's preparation and split: 60% train, 20% validation, 20% test, stratified, with `random_state=0`. Run it.

In [ ]:
from sklearn.model_selection import train_test_split

raw = pd.read_csv("credit_default_taiwan.csv.gz")
clients = raw.rename(columns={"PAY_0": "PAY_1"})                      # 1 = September in every group
clients["EDUCATION"] = clients["EDUCATION"].replace({0: 4, 5: 4, 6: 4})  # undocumented codes -> "others"
STATUS = [f"PAY_{m}" for m in range(1, 7)]
AMOUNTS = ["LIMIT_BAL"] + [f"BILL_AMT{m}" for m in range(1, 7)] + [f"PAY_AMT{m}" for m in range(1, 7)]
CATEGORIES = ["EDUCATION"] + STATUS
FEATURES = CATEGORIES + AMOUNTS        # 20 inputs; SEX, MARRIAGE and AGE are never inputs
TARGET = "default"

rest, test = train_test_split(clients, test_size=0.2, stratify=clients[TARGET], random_state=0)
train, valid = train_test_split(rest, test_size=0.25, stratify=rest[TARGET], random_state=0)
y_train, y_valid, y_test = (part[TARGET].to_numpy() for part in (train, valid, test))
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} {len(part):6d} clients, {part[TARGET].sum():5d} defaults ({part[TARGET].mean():.1%})")

> **Check.** You should see 18,000 / 6,000 / 6,000 clients, each with 22.1% defaults.

## 2. The score and the cost

Every model is judged in two ways. **Average precision (AP)** says how well the scores put the defaults at the top of the list, over all thresholds. The **cost** counts what the decision costs: 5 for each missed default and 1 for each false alarm, at the cheapest threshold found on the validation set. Run the cell. It prints nothing.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

MISS_COST, ALARM_COST = 5, 1
THRESHOLDS = np.round(np.arange(0.01, 1.0, 0.01), 2)


def cost(y, flags):
    """5 for each missed default, 1 for each false alarm."""
    y, flags = np.asarray(y, bool), np.asarray(flags, bool)
    return int(MISS_COST * (~flags & y).sum() + ALARM_COST * (flags & ~y).sum())


def cheapest(y, scores):
    """The threshold with the lowest total cost, and that cost."""
    costs = [cost(y, scores >= t) for t in THRESHOLDS]
    i = int(np.argmin(costs))
    return float(THRESHOLDS[i]), int(costs[i])


def metrics(y, scores):
    return {"ap": round(float(average_precision_score(y, scores)), 4),
            "auc": round(float(roc_auc_score(y, scores)), 4)}

## 3. The models to beat

Two baselines: today's rule ("call every client who was late in September") and the machine learning course's logistic regression pipeline. Run the cell.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def signed_log(values):
    return np.sign(values) * np.log1p(np.abs(values))


def preprocessor(log=True, scale=True, codes_as_categories=True):
    """The machine learning course's preparation: codes one-hot, amounts signed log, then scaled."""
    steps = [("log", FunctionTransformer(signed_log))] if log else []
    steps += [("scale", StandardScaler())] if scale else []
    steps = steps or [("keep", "passthrough")]
    if not codes_as_categories:
        return ColumnTransformer([("all", Pipeline(steps), FEATURES)])
    return ColumnTransformer([
        ("codes", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=False),
         CATEGORIES),
        ("amounts", Pipeline(steps), AMOUNTS)])


logistic = Pipeline([("prepare", preprocessor()), ("model", LogisticRegression(max_iter=2000))])
logistic.fit(train[FEATURES], y_train)
logistic_valid = logistic.predict_proba(valid[FEATURES])[:, 1]
rule_valid = (valid["PAY_1"] >= 1).to_numpy()
print("rule: late in September  ", metrics(y_valid, rule_valid.astype(float)), "cost", cost(y_valid, rule_valid))
print("logistic regression      ", metrics(y_valid, logistic_valid), "cheapest threshold and cost",
      cheapest(y_valid, logistic_valid))

> **Check.** The rule has an AP of about 0.362 and costs 3,934. Logistic regression has an AP of about 0.555, and its cheapest threshold 0.17 costs 3,355.

## 4. Gradient boosting, as it is

Gradient boosting reads the 20 columns as they are. Trees split on any value, so a code like −2 or a bill of NT$900,000 needs no preparation.

> **Predict.** Will boosting with its default settings beat logistic regression's AP of 0.555? By how much?

In [ ]:
import time
from sklearn.ensemble import HistGradientBoostingClassifier


def fit_boosting(params, seed):
    """Gradient boosting on the 20 columns as they are: no one-hot, no log, no scaling."""
    model = HistGradientBoostingClassifier(random_state=seed, **params)
    start = time.perf_counter()
    model.fit(train[FEATURES], y_train)
    seconds = time.perf_counter() - start
    return model, model.predict_proba(valid[FEATURES])[:, 1], seconds


booster, booster_valid, seconds = fit_boosting({}, seed=0)
print("boosting, defaults", metrics(y_valid, booster_valid), f"| {booster.n_iter_} trees in {seconds:.2f} s")

> **Check.** You should see an AP of about 0.559 and 45 trees. The time depends on your computer.

## 5. The network's inputs

A network multiplies every input by a weight, so the inputs must be on similar scales, and codes must not look like amounts. The network uses the same preparation as logistic regression. It also needs a **stopping set** for early stopping: 10% of train, which the network never trains on. Boosting holds out 10% of train in the same way, inside `fit()`. Run the cell.

In [ ]:
fit_part, stop_part = train_test_split(train, test_size=0.1, stratify=train[TARGET], random_state=0)


def tensors(prepare, frames):
    """Prepared columns as float32 tensors, one per frame."""
    return [torch.tensor(prepare.transform(f[FEATURES]), dtype=torch.float32) for f in frames]


prepare = preprocessor().fit(fit_part[FEATURES])   # learn the scaling from the fitting rows only
x_fit, x_stop, x_valid, x_test = tensors(prepare, (fit_part, stop_part, valid, test))
y_fit = torch.tensor(fit_part[TARGET].to_numpy())
y_stop = torch.tensor(stop_part[TARGET].to_numpy())
for name, x in [("x_fit", x_fit), ("x_stop", x_stop), ("x_valid", x_valid), ("x_test", x_test)]:
    show_shape(name, x)
print("first client, first 8 prepared numbers:", [round(v, 2) for v in x_fit[0, :8].tolist()])

> **Check.** You should see 16,200 rows to fit and 1,800 to stop, each with the same number of columns.

> **Your turn.** How many prepared columns does the network read? Put the number in `n_inputs` (use a tensor's shape), then run the check cell.

In [ ]:
n_inputs = x_fit.shape[1]
print(n_inputs)

In [ ]:
expect('n_inputs', n_inputs, 55)

## 6. The network

`TableNet` is the network of this course, with a table in place of a bag of words: linear layers, ReLU, dropout, and 2 scores at the end (pays, defaults). Run the cell.

In [ ]:
class TableNet(nn.Module):
    """A network for a table: (Linear -> ReLU -> Dropout) x depth, then 2 scores."""

    def __init__(self, n_inputs, hidden=64, depth=2, dropout=0.2):
        super().__init__()
        layers, width = [], n_inputs
        for _ in range(depth):
            layers += [nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)]
            width = hidden
        layers.append(nn.Linear(width, 2))           # 2 scores: pays, defaults
        self.layers = nn.Sequential(*layers)

    def forward(self, x):                            # x: (batch, n_inputs) -> (batch, 2)
        return self.layers(x)


torch.manual_seed(0)
model = TableNet(x_fit.shape[1])
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))
show_shape("scores", model(x_fit[:8]))

> **Check.** You should see 7,874 parameters and the scores shape `(8, 2)`.

## 7. Train with early stopping

The training loop is the course's five steps. After each epoch, the loss on the stopping set decides: a new best epoch keeps a copy of the weights; 10 epochs without a new best stop the training, and the best weights come back. Run the cell. It takes a few seconds.

In [ ]:
import copy


def predict(model, x):
    """Probability of default, with dropout off and no gradients."""
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(x), dim=1)[:, 1].numpy()


def train_network(model, x_fit, y_fit, x_stop, y_stop, lr=1e-3, weight_decay=1e-4, batch_size=256,
                  max_epochs=200, patience=10, seed=0):
    """The course's five steps per mini-batch; early stopping on the stopping set."""
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    best_loss, best_state, best_epoch, waited, history = float("inf"), None, 0, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(x_fit))
        total = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            optimizer.zero_grad()                          # 1. forget the last batch's gradients
            loss = loss_fn(model(x_fit[idx]), y_fit[idx])  # 2-3. forward pass, how wrong?
            loss.backward()                                # 4. backward pass: gradients
            optimizer.step()                               # 5. update the weights
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            stop_loss = loss_fn(model(x_stop), y_stop).item()
        history.append({"epoch": epoch, "train_loss": round(total / len(x_fit), 4), "stop_loss": round(stop_loss, 4)})
        if stop_loss < best_loss - 1e-4:                   # a new best: keep a copy of the weights
            best_loss, best_state, best_epoch, waited = stop_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)                      # early stopping: back to the best epoch
    return history, best_epoch


history, best_epoch = train_network(model, x_fit, y_fit, x_stop, y_stop, seed=0)
print(f"best epoch {best_epoch}, stopped after {len(history)} | valid", metrics(y_valid, predict(model, x_valid)))

> **Check.** You should see `best epoch 7, stopped after 17` and an AP of about 0.550.

Now draw the two curves. The grey line marks the best epoch.

In [ ]:
import matplotlib.pyplot as plt
epochs = [row["epoch"] for row in history]
plt.figure(figsize=(6, 3.2))
plt.plot(epochs, [row["train_loss"] for row in history], "--", label="training loss")
plt.plot(epochs, [row["stop_loss"] for row in history], "-o", ms=3, label="stopping-set loss")
plt.axvline(best_epoch, color="grey", lw=0.8)
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.legend(); plt.show()

## 8. The failure: raw columns into the network

`run_network()` puts the steps of sections 5 to 7 in one function. The cell first trains the network three times on the **raw** columns: no log, no scaling, codes as numbers.

> **Predict.** Boosting did not need any preparation. Will the network?

In [ ]:
def run_network(cfg, seed, prep_kwargs=None):
    """Prepare, build, train with early stopping and score one network. Returns the result and the parts."""
    prepare = preprocessor(**(prep_kwargs or {})).fit(fit_part[FEATURES])
    x_fit, x_stop, x_valid, x_test = tensors(prepare, (fit_part, stop_part, valid, test))
    torch.manual_seed(seed)
    model = TableNet(x_fit.shape[1], cfg["hidden"], cfg["depth"], cfg["dropout"])
    start = time.perf_counter()
    history, best_epoch = train_network(model, x_fit, y_fit, x_stop, y_stop, seed=seed)
    seconds = time.perf_counter() - start
    result = metrics(y_valid, predict(model, x_valid)) | {"seed": seed, "seconds": round(seconds, 2),
                                                          "best_epoch": best_epoch}
    return result, (model, prepare, x_test)


START = {"hidden": 64, "depth": 2, "dropout": 0.2}
for seed in range(3):
    result, _ = run_network(START, seed, {"log": False, "scale": False, "codes_as_categories": False})
    print("raw columns:", result)

> **Check.** All three runs give an AP of about 0.221, the share of defaults, and a ROC AUC of about 0.50: the network learned nothing. A credit limit of 800,000 next to a status code of 2 swamps the weighted sums.

> **Your turn.** Change one thing: scale the columns but keep the codes as numbers and skip the log (`{"log": False, "scale": True, "codes_as_categories": False}`), with `START` and seed 0. Put the validation AP in `scaled_ap`.

In [ ]:
scaled_ap = run_network(START, 0, {"log": False, "scale": True, "codes_as_categories": False})[0]["ap"]
print(scaled_ap)

In [ ]:
expect('scaled_ap', scaled_ap, 0.5472, tolerance=0.01)

**What this tells us.** Scaling is the step that matters most. One-hot codes and the log help a little more.

## 9. The same tuning budget for both

Each model gets 8 settings, and each setting runs with 3 seeds, because one network run can be lucky. The score is the median validation AP. Run the cell. It takes about 20 seconds.

In [ ]:
NET_GRID = [{"hidden": h, "depth": d, "dropout": p} for h in (64, 256) for d in (1, 2) for p in (0.0, 0.3)]
BOOST_GRID = [{"learning_rate": r, "max_leaf_nodes": n, "min_samples_leaf": m}
              for r in (0.05, 0.1) for n in (15, 31) for m in (20, 100)]
rows = []
for cfg in NET_GRID:                     # 8 settings x 3 seeds = 24 networks
    aps = [run_network(cfg, seed)[0]["ap"] for seed in (0, 1, 2)]
    rows.append({"model": "network", "setting": str(cfg), "median AP": np.median(aps), "lowest": min(aps)})
for cfg in BOOST_GRID:                   # 8 settings x 3 seeds = 24 boosted models
    aps = [metrics(y_valid, fit_boosting(cfg, seed)[1])["ap"] for seed in (0, 1, 2)]
    rows.append({"model": "boosting", "setting": str(cfg), "median AP": np.median(aps), "lowest": min(aps)})
tuning = pd.DataFrame(rows)
print(tuning.sort_values(["model", "median AP"], ascending=False).groupby("model").head(3).to_string(index=False))
print(tuning.groupby("model")["median AP"].agg(["min", "max"]))

> **Check.** The best network setting is 256 hidden neurons, 1 layer, no dropout (median 0.560). The best boosting setting has a learning rate of 0.05, 15 leaves and at least 100 clients per leaf (median 0.566). The network's 8 settings spread from 0.549 to 0.560; boosting's from 0.559 to 0.566.

## 10. Ten seeds each

The chosen recipes now run with 10 seeds each. Networks change with the seed (the starting weights and the batch order); boosting changes too, because the seed chooses its held-out 10%. Run the cell. It takes about 15 seconds.

In [ ]:
best_net = {"hidden": 256, "depth": 1, "dropout": 0.0}
best_boost = {"learning_rate": 0.05, "max_leaf_nodes": 15, "min_samples_leaf": 100}
final, parts = {"network": [], "boosting": []}, {"network": {}, "boosting": {}}
for seed in range(10):
    result, (model, prepare, x_test) = run_network(best_net, seed)
    final["network"].append(result)
    parts["network"][seed] = (model, prepare, predict(model, x_test))
    booster, scores, seconds = fit_boosting(best_boost, seed)
    final["boosting"].append(metrics(y_valid, scores) | {"seed": seed, "seconds": round(seconds, 2)})
    parts["boosting"][seed] = (booster, None, booster.predict_proba(test[FEATURES])[:, 1])
for name, results in final.items():
    table = pd.DataFrame(results)
    print(f"{name:9s} valid AP median {table['ap'].median():.3f} (lowest {table['ap'].min():.3f}, "
          f"highest {table['ap'].max():.3f}) | fit time median {table['seconds'].median():.2f} s")

> **Check.** Network: median 0.561 (0.551 to 0.564). Boosting: median 0.567 (0.558 to 0.571).

> **Your turn.** Put the network's median validation AP over the 10 seeds into `network_median`. Use `final["network"]`.

In [ ]:
network_median = float(np.median([r["ap"] for r in final["network"]]))
print(network_median)

In [ ]:
expect('network_median', network_median, 0.5614, tolerance=0.005)

## 11. The test set, once

Everything is chosen. Now each model meets the 6,000 test clients once, with the threshold that each seed chose on validation. Run the cell.

In [ ]:
logistic_test = logistic.predict_proba(test[FEATURES])[:, 1]
print(f"{'rule: late in September':28s} AP   -    cost {cost(y_test, test['PAY_1'].to_numpy() >= 1)}")
print(f"{'logistic regression':28s} AP {metrics(y_test, logistic_test)['ap']:.3f} "
      f"cost {cost(y_test, logistic_test >= cheapest(y_valid, logistic_valid)[0])}")
for name in ("boosting", "network"):
    costs, aps = [], []
    for seed, (model, prepare, test_scores) in parts[name].items():
        valid_scores = (model.predict_proba(valid[FEATURES])[:, 1] if prepare is None
                        else predict(model, tensors(prepare, (valid,))[0]))
        threshold = cheapest(y_valid, valid_scores)[0]          # chosen on valid, as before
        costs.append(cost(y_test, test_scores >= threshold))
        aps.append(metrics(y_test, test_scores)["ap"])
    print(f"{name + ', median of 10 seeds':28s} AP {np.median(aps):.3f} cost {np.median(costs):.0f} "
          f"(seeds: {min(costs)} to {max(costs)})")

> **Check.** Boosting's median test AP is about 0.562, the network's 0.556, logistic regression's 0.549. The costs are close: about 3,362, 3,342 and 3,346, against 3,852 for the rule.

## 12. Is the difference real?

The **bootstrap** draws 6,000 test clients with replacement, 1,000 times, and measures the difference each time. Run the cell.

In [ ]:
rng = np.random.default_rng(0)
boost_scores, net_scores = parts["boosting"][0][2], parts["network"][0][2]   # seed 0 of each
differences = []
for _ in range(1000):
    idx = rng.integers(0, len(y_test), len(y_test))          # 6,000 test clients, drawn with replacement
    differences.append(average_precision_score(y_test[idx], boost_scores[idx])
                       - average_precision_score(y_test[idx], net_scores[idx]))
low, high = np.percentile(differences, [2.5, 97.5])
point = average_precision_score(y_test, boost_scores) - average_precision_score(y_test, net_scores)
print(f"boosting minus network, test AP: {point:+.4f}, 95% interval {low:+.4f} to {high:+.4f}")
print(f"share of resamples where boosting is ahead: {np.mean(np.array(differences) > 0):.0%}")

> **Check.** The point difference is about +0.008, and the 95% interval runs from about −0.004 to +0.020. It includes 0.

## 13. Which columns matter?

**Permutation importance** works for any model: shuffle one column and measure how much AP drops. Run the cell. It takes about 5 seconds.

In [ ]:
def permutation_importance(score, frame, y, repeats=5, seed=0):
    """How much AP drops when one column is shuffled (its link to the target is broken)."""
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, score(frame))
    drops = {}
    for column in FEATURES:
        losses = []
        for _ in range(repeats):
            shuffled = frame.copy()
            shuffled[column] = rng.permutation(shuffled[column].to_numpy())
            losses.append(base - average_precision_score(y, score(shuffled)))
        drops[column] = round(float(np.mean(losses)), 4)
    return pd.Series(drops).sort_values(ascending=False)


boost0 = parts["boosting"][0][0]
net0, prepare0 = parts["network"][0][0], parts["network"][0][1]
importance = pd.DataFrame({
    "boosting": permutation_importance(lambda f: boost0.predict_proba(f[FEATURES])[:, 1], valid, y_valid),
    "network": permutation_importance(lambda f: predict(net0, tensors(prepare0, (f,))[0]), valid, y_valid)})
print(importance.sort_values("boosting", ascending=False).head(6))

> **Check.** Both models depend most on `PAY_1`, the status in September (a drop of about 0.21 and 0.15).

## 14. One new client, end to end

A made-up client: two months late in September and August, a bill close to the limit of NT$50,000, and no payment in September. Run the cell.

In [ ]:
new_client = pd.DataFrame([{
    "LIMIT_BAL": 50_000, "EDUCATION": 2,
    "PAY_1": 2, "PAY_2": 2, "PAY_3": 0, "PAY_4": 0, "PAY_5": 0, "PAY_6": 0,
    "BILL_AMT1": 48_200, "BILL_AMT2": 47_100, "BILL_AMT3": 46_000,
    "BILL_AMT4": 30_500, "BILL_AMT5": 29_800, "BILL_AMT6": 29_000,
    "PAY_AMT1": 0, "PAY_AMT2": 1_500, "PAY_AMT3": 1_600, "PAY_AMT4": 1_200, "PAY_AMT5": 1_100, "PAY_AMT6": 1_000}])
for name in ("boosting", "network"):
    scores = [float(model.predict_proba(new_client[FEATURES])[0, 1]) if prepare is None
              else float(predict(model, tensors(prepare, (new_client,))[0])[0])
              for model, prepare, _ in parts[name].values()]
    print(f"{name:9s} seed 0: {scores[0]:.3f} | 10 seeds: {min(scores):.3f} to {max(scores):.3f}")
print(f"logistic           {logistic.predict_proba(new_client[FEATURES])[0, 1]:.3f}")

> **Check.** Every model gives a score well above the threshold of about 0.16, so every model calls this client. The network's 10 seeds disagree by about 0.08 on this one client.

> **Your turn (no check).** Change `PAY_1` and `PAY_2` of the new client to 0 and run the cell again. Which model changes its score the most? Write one sentence.

## 15. Your verdict

Write three sentences for the team: which model you recommend, what the network would cost them, and what would make you test a network again. Compare your answer with the lesson page.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Fine-tune a small transformer](https://learning.nextia-ai.com/courses/deep-learning/m09/fine-tune-a-small-transformer/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m09/deep-network-or-gradient-boosting/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and anonymous: UCI Default of Credit Card Clients (Yeh, 2009), CC BY 4.0.